# 第89章 线性回归与正则化

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 4 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 数据预处理与Pipeline  →  **本章任务：** 线性回归与正则化  →  **下一步：** 逻辑回归分类
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景
**背景引入**：预测一个连续的数值（比如病人的病情进展、某款商品的销量或一地的房价）是日常数据分析里最常见的需求之一，而线性回归正是这类「回归任务」最直观的起点——它把用特征乘上权重再相加得到的数值，去贴近真实结果，先不追求模型有多复杂，先把「能不能预测、误差大概多大」跑通。真正麻烦的往往在第二步：特征一多，模型系数容易被拉得很大，在训练集上表现很好、一到新数据却失效，正则化（Ridge 就是在误差里加一项对系数大小的 L2 惩罚）就是为了压住这种不稳定的系数。这一章就从这两类方法入手，先快速拿到一个能用的模型，再明白什么时候该收一收系数、让模型更稳一些。


## 本章目标

学完本章，你将能够：

- **理解**：理解「线性回归与正则化」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「线性回归与正则化」的关键输出指标。
- **迁移**：能把「线性回归与正则化」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：预测一个连续的量——比如房价、月销售额——我们通常想找一条能“透过特征看结果”的直线。线性回归就是这条“最会讲道理的直线”：它让每个点离线的距离（残差）平方和最小，从而得到一个可解释、可检验的公式。但特征一多，直线容易“太听话”而记住噪声，正则化（Ridge 的 L2）正是为了压住过大的系数而存在。


- 线性回归最小化残差平方和
- Ridge 对大系数施加 L2 惩罚
- MAE 与目标同单位且对极端值较稳健
- R² 小于 0 表示还不如预测测试集均值（打个比方：连“一律猜平均值”这个最呆但稳的答案都不如，说明模型是被数据带偏了，还不如什么都不学。）


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 糖尿病进展回归 | `y.describe()`、`.round()`、`.to_dict()` | Diabetes 数据包含 442 个样本和 10 个标准化生理特征。 | 把相关系数解释成因果效应 |
| OLS 与 Ridge 对比 | `model.fit()`、`model.predict()`、`rows.append()`、`np.linalg.norm()` | 在同一测试集上比较预测误差和系数范数。 | 只报告 R² 不报告误差单位 |


## 例 1｜糖尿病进展回归

Diabetes 数据包含 442 个样本和 10 个标准化生理特征。


<!-- math-foundation:chapter-89 -->
### 数学推导｜线性回归与正则化

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜普通最小二乘只惩罚预测残差。** 矩阵形式为 $\lVert y-X\beta\rVert_2^2$。

**第 2 步｜Ridge 再惩罚大系数。** 目标变为

$$
J(\beta)=\lVert y-X\beta\rVert_2^2+\lambda\lVert\beta\rVert_2^2
$$

**第 3 步｜令梯度为 0。** $-2X^T(y-X\beta)+2\lambda\beta=0$，因此在相应可逆条件下

$$
\hat\beta=(X^TX+\lambda I)^{-1}X^Ty
$$

$\lambda$ 增大使解更稳定、方差通常下降，但偏差会上升；截距通常不参与惩罚。

**把上面的关系收束为本章计算式：**

$$
\hat{y}=\beta_0+x^T\beta,\qquad \min_{\beta}\sum_i(y_i-\hat{y}_i)^2+\lambda\lVert\beta\rVert_2^2
$$

**符号解释：** $\lambda$ 控制 Ridge 正则强度，抑制过大的系数。

**代码对应：** 比较 `LinearRegression` 与 `Ridge(alpha=...)` 的验证误差。

**使用边界：** 系数依赖特征尺度；线性预测关系不等于因果效应。


In [ ]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=78
)
print(X.shape, y.describe().round(1).to_dict())


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练：调整数据切分比例，观察样本数变化**示例 1 用 `test_size=0.25` 把数据切成训练集和测试集。请把切分比例从 `0.25` 改成 `0.40`，再重新运行，然后核对训练集和测试集的样本数量分别变成了多少。运行前先猜一下：切分比例变大后，训练集是变多还是变少？测试集呢？运行后对照打印出的 `X_train2.shape[0]` 和 `X_test2.shape[0]`，看看和你的直觉是否一致，并把它写进注释里。


In [ ]:
try:
    pass
    # 请在下方填写代码：把 test_size 从 0.25 改成 0.40，再观察训练集/测试集样本数变化。#
    # 提示：train_test_split(X, y, test_size=____, random_state=78)X_train2,
    # X_test2, y_train2, y_test2 = train_test_split(    X, y, test_size=0.40,
    # random_state=78)# 请在下方填写代码（打印两个集合的样本数）：print('训练集样本数：',
    # X_train2.shape[0])print('测试集样本数：', X_test2.shape[0])# ——

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜OLS 与 Ridge 对比

在同一测试集上比较预测误差和系数范数。


In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rows = []
for name, model in {
    "OLS": LinearRegression(),
    "Ridge": Ridge(alpha=10),
}.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append(
        [
            name,
            mean_absolute_error(y_test, pred),
            mean_squared_error(y_test, pred) ** 0.5,
            r2_score(y_test, pred),
            np.linalg.norm(model.coef_),
        ]
    )
result = pd.DataFrame(
    rows, columns=["模型", "MAE", "RMSE", "R2", "系数L2范数"]
).set_index("模型")
display(result.round(3))


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = model.predict(X_changed)
print("原始前2个预测：", np.round(model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print("预测变化：", np.round(changed_prediction[:2] - model.predict(X[:2]), 2))


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 把相关系数解释成因果效应
- 只报告 R² 不报告误差单位
- 用测试集选择 alpha 后仍称其为最终测试集
- 线性外推到训练范围之外


## 练习与作业

1. 比较 alpha 为 0.1、1、10、100 的 Ridge
2. 选择测试 RMSE 最低者
3. 观察 alpha 与系数范数关系

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 89.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“比较 alpha 为 0.1、1、10、100 的 Ridge”。
2. **独立完成**：不复制示例代码，完成“选择测试 RMSE 最低者”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“观察 alpha 与系数范数关系”，用一两句话说明你修改了什么。

### 89.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 89.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用线性回归预测连续目标，并通过 Ridge 正则化控制系数规模；用 MAE、RMSE 和 R² 从不同角度评价误差。


### 你已经掌握

- 识别回归任务
- 解释线性模型的系数与截距
- 计算 MAE、RMSE 和 R²
- 理解 Ridge 的 alpha 对偏差和方差的影响


### 需要注意

- 把相关系数解释成因果效应
- 只报告 R² 不报告误差单位
- 用测试集选择 alpha 后仍称其为最终测试集
- 线性外推到训练范围之外


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 参考实现：把 test_size 改成 0.40，其余保持不变。X_train2, X_test2, y_train2, y_test2 =
# train_test_split(    X, y, test_size=0.40,
# random_state=78)print('训练集样本数：', X_train2.shape[0])print('测试集样本数：',


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
ridge_rows = []
for alpha in [0.1, 1, 10, 100]:
    m = Ridge(alpha=alpha).fit(X_train, y_train)
    p = m.predict(X_test)
    ridge_rows.append(
        [alpha, mean_squared_error(y_test, p) ** 0.5, np.linalg.norm(m.coef_)]
    )
practice_result = pd.DataFrame(
    ridge_rows, columns=["alpha", "RMSE", "coef_norm"]
)
display(practice_result.round(3))
